---
# 🎯 **YouTube Watch History — Health Classifier**
---

## **📦 Cell 1 — Install & Import Libraries**

In [ ]:
import json, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from collections import Counter
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import yt_dlp

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ All libraries loaded successfully!')

## **🧹 Cell 2 — Load & Clean Data**

In [ ]:
# ── Load JSON ──────────────────────────────────────────────────────────────
JSON_PATH = 'watchHistory.json'   # ← change path if needed

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print(f'Raw entries loaded: {len(raw):,}')

# ── Parse to DataFrame ─────────────────────────────────────────────────────
records = []
for entry in raw:
    # Skip non-watch events (e.g., 'Viewed' = search/page visits)
    title_raw = entry.get('title', '')
    if not title_raw.startswith('Watched '):
        continue

    # Strip the 'Watched ' prefix
    title = title_raw[len('Watched '):].strip()

    # Channel name (first subtitle if present)
    subs = entry.get('subtitles', [])
    channel = subs[0]['name'].strip() if subs else 'Unknown'

    # Video URL / ID
    url = entry.get('titleUrl', '')
    vid_id = url.split('v=')[-1] if 'v=' in url else ''

    # Timestamp
    ts = entry.get('time', '')
    try:
        dt = datetime.fromisoformat(ts.replace('Z', '+00:00'))
    except Exception:
        dt = None

    records.append({
        'title': title,
        'channel': channel,
        'video_id': vid_id,
        'url': url,
        'timestamp': dt,
    })

df = pd.DataFrame(records)

# ── Feature Engineering ────────────────────────────────────────────────────
df['date']         = df['timestamp'].dt.date
df['hour']         = df['timestamp'].dt.hour
df['day_of_week']  = df['timestamp'].dt.day_name()
df['week']         = df['timestamp'].dt.isocalendar().week.astype(int)
df['month']        = df['timestamp'].dt.month_name()

# Title-level signals
df['title_len']      = df['title'].str.len()
df['is_short']       = df['title'].str.contains('#shorts|#short|#ytshorts', case=False, na=False).astype(int)
df['has_emoji']      = df['title'].str.contains(r'[\U00010000-\U0010ffff]|[\u2600-\u27BF]', na=False).astype(int)
df['caps_ratio']     = df['title'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t),1))
df['exclaim_count']  = df['title'].str.count(r'[!?]')
df['hashtag_count']  = df['title'].str.count(r'#\w+')
df['emoji_count']    = df['title'].str.count(r'[\U00010000-\U0010ffff]|[\u2600-\u27BF]')

# Composite text feature for NLP
df['text_combined'] = df['title'].fillna('') + ' ' + df['channel'].fillna('')

print(f'\nCleaned entries (watch events only): {len(df):,}')
print(f'Date range: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Unique channels: {df["channel"].nunique():,}')

# To display he first few rows of the DataFrame, you can use the following code:
# df.head(5)

## **🏷️ Cell 3 — Rule-Based Auto-Labeling**

In [ ]:
# ── Healthy / Productive keyword lists ────────────────────────────────────
HEALTHY_KEYWORDS = [
    # Education & Learning
    'tutorial', 'explained', 'how to', 'learn', 'course', 'lecture','study', 'science', 'physics', 'math', 'chemistry', 'biology','history', 'geography', 'economics', 'engineering', 'programming','python',
    'javascript', 'machine learning', 'data science', 'ai','algorithm', 'deep learning', 'neural', 'statistics','linear algebra', 'calculus', 'probability', 'computer science','astrophysics', 'quantum mechanics',
    # Self-improvement
    'productivity', 'motivation', 'habits', 'mindset', 'discipline','stoic', 'philosophy', 'psychology', 'book review', 'book summary','self improvement', 'self-improvement', 'ted talk', 'ted-ed',
    'time management', 'focus', 'goal setting', 'success habits','morning routine', 'life lessons', 'decision making',
    # Health & Fitness
    'workout', 'exercise', 'fitness', 'yoga', 'meditation', 'nutrition','diet', 'mental health', 'sleep', 'wellness','cardio', 'strength training', 'home workout', 'weight loss','healthy lifestyle', 'mindfulness', 'breathing exercises',
    # Finance & Career
    'finance', 'investing', 'stock market', 'budget', 'career','interview prep', 'resume', 'side hustle', 'passive income','entrepreneurship', 'startup', 'business strategy','financial literacy', 'personal finance', 'wealth building',
    # News & Knowledge
    'documentary', 'news analysis', 'research', 'paper explained','case study', 'analysis', 'current affairs', 'world affairs','geopolitics', 'economy explained', 'policy', 'education system',
    # Skill building
    'coding', 'design', 'photography', 'music theory', 'drawing','video editing', 'public speaking', 'communication skills','critical thinking', 'problem solving', 'writing skills',
    # Tech & Innovation
    'technology', 'gadgets', 'software development', 'open source','cybersecurity', 'cloud computing', 'blockchain', 'web development','app development', 'devops',
    # Creativity & Thinking
    'creativity', 'innovation', 'ideas', 'brain training','learning techniques', 'memory techniques'
]
HEALTHY_CHANNELS = [
    # Global Education / Science
    '3blue1brown', 'kurzgesagt', 'veritasium', 'vsauce','lex fridman', 'mit opencourseware', 'stanford','ted', 'ted-ed', 'crash course','khan academy', 'freecodecamp', 'sentdex','two minute papers', 'andrew huberman',
    'ali abdaal', 'thomas frank', 'marques brownlee','mark rober', 'wendover productions','real engineering', 'smarter every day','minutephysics', 'numberphile', 'computerphile','fireship', 'traversy media', 'the coding train','corey schafer',
    # Finance / Business
    'graham stephan', 'andrei jikh', 'minority mindset','meet kevin', 'the plain bagel', 'coffeezilla',
    # Self Improvement / Productivity
    'better ideas', 'improvement pill', 'matt d avella','cal newport', 'productivity game',
    # Indian Educational / Tech / Finance (expanded heavily)
    'arjun pandey','apna college','code with harry','love babbar','take u forward','gate smashers','unacademy','byjus','physics wallah','study iq education','wifistudy','examfear education','learn coding','geeksforgeeks','scaler','coding ninjas',
    # Indian Finance / Growth
    'anushka rathod','pranjal kamra','asset yogi','labour law advisor','warikoo','think school','finology','ca rachana ranade',
    # Indian Science / Knowledge
    'bharat ki kahani','soch by mohak mangal','the sham sharma show','the sham sharma show global','sham sharma show','sham sharma show global','study iq','world affairs',
    # Indian Productivity / Lifestyle
    'beerbiceps','ranveer allahbadia','raj shamani','anubhav jain','nitesh rajput',
    # Tech / Programming India
    'tech burner','technical guruji','code basics','hitesh choudhary','chai aur code','akshay saini','programming with mosh','sheryians coding school'
    # Misc high-quality knowledge
    'big think','school of life','coldfusion','poly matter','neo','visualpolitik'
]

# ── Brainrot / Mindless keyword lists ────────────────────────────────────
BASE_BRAINROT_KEYWORDS = [
    '#shorts', '#short', '#ytshorts', '#trending', '#viral','#reels', '#memes', 'meme', 'prank', 'gone wrong', 'gone sexual','exposed', 'roast', 'beef', 'diss track', 'funniest',
    'try not to laugh', 'you wont believe', 'wait for it','watch till end', 'part 1', 'part 2', 'sigma', 'sussy','baka', 'skibidi', 'ohio', 'rizz', 'npc', 'aura',
    'brainrot', 'brain rot', 'edits', 'amv', 'phonk','tiktok', 'compilation', 'satisfying', 'oddly satisfying','mukbang', 'asmr', 'unboxing', 'haul', 'storytime',
    'challenge', 'dare challenge', 'clickbait', 'reaction','reacting to', 'i tried', 'spending 24 hours','caught on camera', 'insane', 'crazy', 'wtf',
    'fails', 'epic fail', 'instant regret', 'best moments','top 10', 'top 5', 'before and after', 'glow up','transformation', 'life hack', 'hack', 'secret trick'
]

# Expand to ~1000 automatically
MODIFIERS = [
    'ultimate', 'best', 'funniest', 'craziest', 'insane','viral', 'trending', '2025', 'new', 'must watch','compilation', 'edition', 'remix', 'loop', 'extended'
]

FORMATS = [
    '{} {}', '{} {} compilation', '{} {} part {}','{} {} reaction', '{} {} challenge','{} {} moments', '{} {} tiktok'
]

BRAINROT_KEYWORDS = set(BASE_BRAINROT_KEYWORDS)

for base in BASE_BRAINROT_KEYWORDS:
    for mod in MODIFIERS:
        for fmt in FORMATS:
            for i in range(1, 6):
                BRAINROT_KEYWORDS.add(fmt.format(mod, base, i))

# Convert back to list
BRAINROT_KEYWORDS = list(BRAINROT_KEYWORDS)

# ── Brainrot / Mindless channels (expanded heavily) ───────────────────────

BASE_BRAINROT_CHANNELS = [
    'neon man shorts', 'the anime gang', 'ajw battle yt',
    'highkeyhateme', 'amir visions','dhruv rathee','the deshbhakt',

    # Meme / Shorts / Viral
    'beluga', 'speed mcqueen', 'mrbeast gaming shorts',
    'sidemen shorts', 'beta squad',
    'daily dose of internet', 'meme zee',
    'instant regret', 'failarmy',

    # TikTok-style repost / edits
    'viral hog', 'trending clips', 'clip central',
    'meme hub', 'editz zone', 'phonk nation',
    'sigma edits', 'aura edits',

    # Anime edits / AMVs
    'anime edits', 'amv heaven', 'weeb central',
    'otaku edits', 'anime viral',

    # Indian brainrot / reels style
    'carryminati shorts', 'triggered insaan shorts',
    'funcho shorts', 'round2hell shorts',
    'indian meme factory', 'desi meme hub',
    'bakchodi tv', 'indian viral clips',

    # Reaction / drama
    'reaction time', 'drama alert',
    'internet anarchist', 'pegasus reacts',

    # Gaming brainrot
    'total gaming shorts', 'techno gamerz shorts',
    'mortal shorts', 'scout shorts',

    # Satisfying / ASMR
    'satisfying world', 'oddly satisfying hub',
    'asmr universe', 'slime videos',
]

# Auto-expand to ~300–400
PREFIXES = [
    'official', 'tv', 'hub', 'central', 'world',
    'daily', 'viral', 'trending', 'zone', 'nation'
]

SUFFIXES = [
    'shorts', 'clips', 'videos', 'compilation',
    'channel', 'media', 'network'
]

BRAINROT_CHANNELS = set(BASE_BRAINROT_CHANNELS)

for base in BASE_BRAINROT_CHANNELS:
    for p in PREFIXES:
        BRAINROT_CHANNELS.add(f"{p} {base}")
    for s in SUFFIXES:
        BRAINROT_CHANNELS.add(f"{base} {s}")

BRAINROT_CHANNELS = list(BRAINROT_CHANNELS)

### **💯 Scoring Function**

In [ ]:
def label_video(row):
    title_l   = row['title'].lower()
    channel_l = row['channel'].lower()
    text_l    = title_l + ' ' + channel_l

    h_score = 0
    b_score = 0

    # Keyword hits
    for kw in HEALTHY_KEYWORDS:
        if kw in text_l:
            h_score += 1
    for kw in BRAINROT_KEYWORDS:
        if kw in text_l:
            b_score += 1

    # Channel hits (strong signal — double weight)
    for ch in HEALTHY_CHANNELS:
        if ch in channel_l:
            h_score += 2
    for ch in BRAINROT_CHANNELS:
        if ch in channel_l:
            b_score += 2

    # Structural signals
    if row['is_short']:
        b_score += 1.5
    if row['caps_ratio'] > 0.4:
        b_score += 1
    if row['emoji_count'] >= 3:
        b_score += 0.5
    if row['exclaim_count'] >= 2:
        b_score += 0.5
    if row['hashtag_count'] >= 3:
        b_score += 1

    # Late-night watch (11PM–4AM) — mild brainrot signal
    if row['hour'] in [23, 0, 1, 2, 3, 4]:
        b_score += 0.3

    if h_score > b_score:
        return 'healthy'
    elif b_score > h_score:
        return 'brainrot'
    else:
        return 'brainrot'  # default to brainrot if tie (conservative)

df['label'] = df.apply(label_video, axis=1)

label_counts = df['label'].value_counts()
print('Label distribution:')
print(label_counts)
print(f'\nHealthy %: {label_counts.get("healthy",0)/len(df)*100:.1f}%')
print(f'Brainrot %: {label_counts.get("brainrot",0)/len(df)*100:.1f}%')

---
## **🤖 Cell 4 — Train ML Classifier**

In [ ]:
# ── Prepare features ───────────────────────────────────────────────────────
X_text = df['text_combined'].fillna('')
y      = (df['label'] == 'brainrot').astype(int)   # 1=brainrot, 0=healthy

X_train_t, X_test_t, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

# ── Build TF-IDF + Logistic Regression Pipeline ────────────────────────────
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=15000,
        sublinear_tf=True,
        strip_accents='unicode',
        analyzer='word',
        min_df=2
    )),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

pipeline.fit(X_train_t, y_train)

# ── Evaluate ───────────────────────────────────────────────────────────────
y_pred  = pipeline.predict(X_test_t)
y_proba = pipeline.predict_proba(X_test_t)[:, 1]

print('=' * 55)
print('       CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred,
                             target_names=['Healthy', 'Brainrot']))

cv_scores = cross_val_score(pipeline, X_text, y,
                             cv=StratifiedKFold(5), scoring='f1')
print(f'5-Fold CV F1 : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'ROC-AUC      : {roc_auc_score(y_test, y_proba):.3f}')

# Predict on full dataset
df['pred_label']     = pipeline.predict(X_text)
df['brainrot_proba'] = pipeline.predict_proba(X_text)[:, 1]
df['pred_class']     = df['pred_label'].map({0: 'healthy', 1: 'brainrot'})

print('\n✅ Model trained & predictions applied to full dataset!')

## **📊 Cell 5 — Overall Health Score (Numerical)**

In [ ]:
# ── Aggregate Health Metrics ───────────────────────────────────────────────
total          = len(df)
n_healthy      = (df['pred_class'] == 'healthy').sum()
n_brainrot     = (df['pred_class'] == 'brainrot').sum()
pct_healthy    = n_healthy / total * 100
pct_brainrot   = n_brainrot / total * 100
avg_brainrot_p = df['brainrot_proba'].mean() * 100
health_score   = round(100 - avg_brainrot_p, 1)   # 0–100 composite score

# Short-form content stats
shorts_df      = df[df['is_short'] == 1]
pct_shorts     = len(shorts_df) / total * 100

# Late-night sessions
late_night     = df[df['hour'].isin([23,0,1,2,3])]
pct_late       = len(late_night) / total * 100

# Top healthy & brainrot channels
top_healthy_ch  = (df[df['pred_class']=='healthy']['channel']
                   .value_counts().head(5))
top_brainrot_ch = (df[df['pred_class']=='brainrot']['channel']
                   .value_counts().head(5))

# Rating
if health_score >= 70:
    rating = '🟢 HEALTHY'
elif health_score >= 45:
    rating = '🟡 MIXED'
else:
    rating = '🔴 BRAINROT DOMINANT'

print('=' * 60)
print('        🎯 YOUR YOUTUBE HEALTH REPORT')
print('=' * 60)
print(f'  Total Videos Analyzed  : {total:,}')
print(f'  Date Range              : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print('-' * 60)
print(f'  🟢 Healthy / Productive : {n_healthy:,}  ({pct_healthy:.1f}%)')
print(f'  🔴 Brainrot / Mindless  : {n_brainrot:,}  ({pct_brainrot:.1f}%)')
print(f'  📱 Short-form (Shorts)  : {len(shorts_df):,}  ({pct_shorts:.1f}%)')
print(f'  🌙 Late-Night Watches   : {len(late_night):,}  ({pct_late:.1f}%)')
print('-' * 60)
print(f'  ⭐ HEALTH SCORE         : {health_score}/100')
print(f'  📋 VERDICT              : {rating}')
print('=' * 60)
print()
print('  Top Healthy Channels:')
for ch, cnt in top_healthy_ch.items():
    print(f'    {ch:<35} {cnt:>4} videos')
print()
print('  Top Brainrot Channels:')
for ch, cnt in top_brainrot_ch.items():
    print(f'    {ch:<35} {cnt:>4} videos')

## **💾 Cell 6 — Export Results**

In [ ]:
# Save classified dataset
export_cols = ['title','channel','timestamp','date','hour','day_of_week',
               'is_short','label','pred_class','brainrot_proba']
df[export_cols].to_csv('youtube_classified.csv', index=False)
print('✅ Classified data saved → youtube_classified.csv')

# Save model
import pickle
with open('youtube_classifier_model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print('✅ Model saved → youtube_classifier_model.pkl')

# Summary stats to JSON
summary = {
    'total_videos': int(total),'healthy_count': int(n_healthy),'brainrot_count': int(n_brainrot),'healthy_pct': round(pct_healthy, 2),'brainrot_pct': round(pct_brainrot, 2),
    'health_score': health_score,'verdict': rating,'shorts_pct': round(pct_shorts, 2),'late_night_pct': round(pct_late, 2),'roc_auc': round(roc_auc_score(y_test, y_proba), 4)
}
with open('health_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print('✅ Summary saved → health_summary.json')
print()
print(json.dumps(summary, indent=2))

## **🔎 Cell 7 — Classify New Videos based on URL**

In [ ]:
# Classify a NEW video title using function
def classify_new(title, channel=''):
    text = f'{title} {channel}'
    prob = pipeline.predict_proba([text])[0]
    pred = pipeline.predict([text])[0]
    label = '🔴 BRAINROT' if pred == 1 else '🟢 HEALTHY'
    print(f'Title   : {title}')
    print(f'Channel : {channel if channel else "(not provided)"}')
    print(f'Result  : {label}')
    print(f'Healthy prob  : {prob[0]*100:.1f}%')
    print(f'Brainrot prob : {prob[1]*100:.1f}%')

# Extract title and channel from a YouTube URL using yt-dlp
def get_youtube_info_ytdlp(url):
    with yt_dlp.YoutubeDL({'quiet': True, 'no_warnings': True}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {"title": info['title'], "channel": info['channel']}
    
# Classify a new video by URL    
info = get_youtube_info_ytdlp('https://youtu.be/3LRZRSIh_KE?si=YA9jM3MPHtkvgonl')
classify_new(info['title'], info['channel'])

---
# **🎞️ Visualization Section:**
---

In [ ]:
# ── Color palette ─────────────────────────────────────────────────────────
HEALTHY_CLR  = '#2ECC71'
BRAINROT_CLR = '#E74C3C'
BG           = '#0F0F0F'
CARD         = '#1A1A2E'
TEXT         = '#ECF0F1'

### FIG 1: Gauge + Pie Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)

# — Pie chart —
ax1 = axes[0]
ax1.set_facecolor(BG)
sizes  = [pct_healthy, pct_brainrot]
colors = [HEALTHY_CLR, BRAINROT_CLR]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=['Healthy / Productive', 'Brainrot / Mindless'],
    colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor=BG, linewidth=2),
    textprops=dict(color=TEXT, fontsize=11)
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(12)
ax1.set_title('Overall Content Split', color=TEXT, fontsize=14, pad=12)

# — Gauge / Speedometer —
ax2 = axes[1]
ax2.set_facecolor(BG)
ax2.set_xlim(-1.4, 1.4); ax2.set_ylim(-0.3, 1.4)
ax2.axis('off')

# Background arc (grey)
theta = np.linspace(np.pi, 0, 300)
ax2.plot(np.cos(theta), np.sin(theta), linewidth=22, color='#333333', solid_capstyle='butt')

# Colored arc up to health_score
score_frac = health_score / 100
theta_fill = np.linspace(np.pi, np.pi - score_frac * np.pi, 300)
grad_color = HEALTHY_CLR if health_score >= 70 else ('#F39C12' if health_score >= 45 else BRAINROT_CLR)
ax2.plot(np.cos(theta_fill), np.sin(theta_fill), linewidth=22, color=grad_color, solid_capstyle='butt')

# Needle
needle_angle = np.pi - score_frac * np.pi
ax2.annotate('', xy=(0.75*np.cos(needle_angle), 0.75*np.sin(needle_angle)),
             xytext=(0,0),
             arrowprops=dict(arrowstyle='->', color='white', lw=2.5))
ax2.plot(0, 0, 'o', color='white', markersize=8, zorder=5)

# Labels
ax2.text(-1.2, -0.15, '0\nBrainrot', color=BRAINROT_CLR, ha='center', fontsize=9)
ax2.text( 1.2, -0.15, '100\nHealthy', color=HEALTHY_CLR, ha='center', fontsize=9)
ax2.text( 0,  0.45, f'{health_score}', color='white', ha='center',
          fontsize=42, fontweight='bold')
ax2.text( 0,  0.12, 'Health Score', color=TEXT, ha='center', fontsize=13)
ax2.text( 0, -0.18, rating, color=grad_color, ha='center', fontsize=13, fontweight='bold')
ax2.set_title('Your YouTube Health Gauge', color=TEXT, fontsize=14)

plt.suptitle('🎯 YouTube Watch History — Health Dashboard', color=TEXT,
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(export_dir / 'fig1_gauge_pie.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 2: Daily trend of Brainrot probability

In [ ]:
daily = (df.groupby('date')
           .agg(brainrot_pct=('brainrot_proba', lambda x: x.mean()*100),
                count=('title', 'count'))
           .reset_index())
daily['date'] = pd.to_datetime(daily['date'])
daily['rolling7'] = daily['brainrot_pct'].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(15, 5), facecolor=BG)
ax.set_facecolor(BG)
ax.fill_between(daily['date'], daily['brainrot_pct'], alpha=0.25, color=BRAINROT_CLR)
ax.plot(daily['date'], daily['brainrot_pct'], color=BRAINROT_CLR, alpha=0.5,
        linewidth=1, label='Daily Brainrot %')
ax.plot(daily['date'], daily['rolling7'], color='#F39C12', linewidth=2.5,
        label='7-Day Moving Avg')
ax.axhline(50, color='white', linestyle='--', linewidth=0.8, alpha=0.4, label='50% threshold')
ax.set_ylim(0, 100)
ax.set_xlabel('Date', color=TEXT); ax.set_ylabel('Brainrot %', color=TEXT)
ax.set_title('📅 Daily Brainrot Score Over Time', color=TEXT, fontsize=14)
ax.tick_params(colors=TEXT)
for spine in ax.spines.values(): spine.set_edgecolor('#444')
ax.legend(facecolor=CARD, labelcolor=TEXT, edgecolor='#444')
plt.tight_layout()
plt.savefig('fig2_daily_trend.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 3: Hourly heatmap — When is Brainrot consumed?

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat_data = (df.groupby(['day_of_week','hour'])['brainrot_proba']
               .mean().unstack(fill_value=0.5))
heat_data = heat_data.reindex([d for d in dow_order if d in heat_data.index])

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
ax.set_facecolor(BG)
sns.heatmap(heat_data, ax=ax, cmap='RdYlGn_r', vmin=0, vmax=1,
            linewidths=0.5, linecolor='#1a1a1a',
            cbar_kws={'label': 'Brainrot Probability', 'shrink': 0.8})
ax.set_title('🕐 Hourly Brainrot Heatmap (by Day of Week)', color=TEXT, fontsize=14)
ax.set_xlabel('Hour of Day (UTC)', color=TEXT)
ax.set_ylabel('', color=TEXT)
ax.tick_params(colors=TEXT)
plt.tight_layout()
plt.savefig('fig3_heatmap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 4: Top 15 Channels — Healthy vs Brainrot (Bar Graph Representation)

In [ ]:
ch_stats = (df.groupby('channel')
              .agg(total=('title','count'),
                   brainrot_avg=('brainrot_proba','mean'))
              .query('total >= 5')
              .sort_values('total', ascending=False)
              .head(20))

ch_stats = ch_stats.sort_values('brainrot_avg', ascending=True)
colors_ch = [HEALTHY_CLR if b < 0.5 else BRAINROT_CLR for b in ch_stats['brainrot_avg']]

fig, ax = plt.subplots(figsize=(13, 8), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(ch_stats.index, ch_stats['brainrot_avg']*100,
               color=colors_ch, edgecolor='none', height=0.7)
ax.axvline(50, color='white', linewidth=1.2, linestyle='--', alpha=0.5)
for bar, (_, row) in zip(bars, ch_stats.iterrows()):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
            f"{row['total']} videos", va='center', color=TEXT, fontsize=8.5)
ax.set_xlabel('Avg Brainrot Probability (%)', color=TEXT)
ax.set_title('📺 Top Channels — Brainrot Score (≥5 videos)', color=TEXT, fontsize=13)
ax.tick_params(colors=TEXT)
for spine in ax.spines.values(): spine.set_edgecolor('#444')
ax.set_xlim(0, 115)
healthy_patch  = mpatches.Patch(color=HEALTHY_CLR, label='Healthy')
brainrot_patch = mpatches.Patch(color=BRAINROT_CLR, label='Brainrot')
ax.legend(handles=[healthy_patch, brainrot_patch],
          facecolor=CARD, labelcolor=TEXT, edgecolor='#444')
plt.tight_layout()
plt.savefig('fig4_channels.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 5: Word Clouds — Healthy vs Brainrot titles

In [ ]:
healthy_text  = ' '.join(df[df['pred_class']=='healthy']['title'].fillna(''))
brainrot_text = ' '.join(df[df['pred_class']=='brainrot']['title'].fillna(''))

def make_wc(text, color, bg):
    return WordCloud(
        width=700, height=400, background_color=bg,
        colormap=('Greens' if color=='green' else 'Reds'),
        max_words=120, collocations=False,
        stopwords={'the','a','an','is','in','of','and','to','for',
                   'you','your','this','that','with','was','are',
                   'Watched','watch','be','on','at','by','as',
                   'it','its','i','my','me','we','our'}
    ).generate(text)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)
for ax, text, color, label in [
    (ax1, healthy_text,  'green', '🟢 Healthy Content'),
    (ax2, brainrot_text, 'red',   '🔴 Brainrot Content')
]:
    ax.set_facecolor(BG)
    wc = make_wc(text, color, '#0d1117')
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud — {label}', color=TEXT, fontsize=13)

plt.tight_layout()
plt.savefig('fig5_wordclouds.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 6: Model diagnostics — Confusion Matrix + ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Healthy', 'Brainrot'])
disp.plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title('Confusion Matrix', color=TEXT, fontsize=13)
ax1.set_facecolor(BG)
ax1.tick_params(colors=TEXT)
ax1.xaxis.label.set_color(TEXT)
ax1.yaxis.label.set_color(TEXT)

# ROC curve
ax2.set_facecolor(BG)
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val = roc_auc_score(y_test, y_proba)
ax2.plot(fpr, tpr, color='#3498DB', linewidth=2.5, label=f'AUC = {auc_val:.3f}')
ax2.plot([0,1],[0,1], 'w--', linewidth=1, alpha=0.5)
ax2.fill_between(fpr, tpr, alpha=0.15, color='#3498DB')
ax2.set_xlabel('False Positive Rate', color=TEXT)
ax2.set_ylabel('True Positive Rate', color=TEXT)
ax2.set_title('ROC Curve', color=TEXT, fontsize=13)
ax2.legend(facecolor=CARD, labelcolor=TEXT, edgecolor='#444')
ax2.tick_params(colors=TEXT)
for spine in ax2.spines.values(): spine.set_edgecolor('#444')

plt.suptitle('🧪 Model Performance Diagnostics', color=TEXT, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_model_diagnostics.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### FIG 7: Weekly health trend (stacked bar)

In [ ]:
weekly = (df.groupby(['week','pred_class'])
            .size().unstack(fill_value=0)
            .reset_index())

fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
ax.set_facecolor(BG)
bottom = np.zeros(len(weekly))
for cls, clr in [('healthy', HEALTHY_CLR), ('brainrot', BRAINROT_CLR)]:
    if cls in weekly.columns:
        ax.bar(weekly['week'], weekly[cls], bottom=bottom,
               color=clr, label=cls.title(), alpha=0.9, width=0.7)
        bottom += weekly[cls].values
ax.set_xlabel('Week of Year', color=TEXT)
ax.set_ylabel('Videos Watched', color=TEXT)
ax.set_title('📆 Weekly Watch Volume — Healthy vs Brainrot', color=TEXT, fontsize=13)
ax.legend(facecolor=CARD, labelcolor=TEXT, edgecolor='#444')
ax.tick_params(colors=TEXT)
for spine in ax.spines.values(): spine.set_edgecolor('#444')
plt.tight_layout()
plt.savefig('fig7_weekly.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# FIG 8: Top TF-IDF feature importance

In [ ]:
tfidf    = pipeline.named_steps['tfidf']
clf_lr   = pipeline.named_steps['clf']
features = tfidf.get_feature_names_out()
coefs    = clf_lr.coef_[0]

top_n  = 20
top_idx_brainrot = np.argsort(coefs)[-top_n:][::-1]
top_idx_healthy  = np.argsort(coefs)[:top_n]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), facecolor=BG)
for ax, idx, color, title in [
    (ax1, top_idx_brainrot, BRAINROT_CLR, '🔴 Top Brainrot Keywords'),
    (ax2, top_idx_healthy,  HEALTHY_CLR,  '🟢 Top Healthy Keywords'),
]:
    ax.set_facecolor(BG)
    kws  = features[idx]
    vals = np.abs(coefs[idx])
    ax.barh(range(top_n), vals[::-1], color=color, alpha=0.85, edgecolor='none')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(kws[::-1], color=TEXT, fontsize=9)
    ax.set_xlabel('Coefficient Magnitude', color=TEXT)
    ax.set_title(title, color=TEXT, fontsize=12)
    ax.tick_params(colors=TEXT)
    for spine in ax.spines.values(): spine.set_edgecolor('#444')

plt.suptitle('🔍 Model Feature Importance — What Drives Classification?',
             color=TEXT, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_features.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## **Optional: 🔍 Inspect Individual Predictions**
---

In [ ]:
# Most confident BRAINROT predictions
print('🔴 Top 10 Most Brainrot Videos:')
print(df.nlargest(10, 'brainrot_proba')[['title','channel','brainrot_proba']]
        .rename(columns={'brainrot_proba':'Brainrot Score'})
        .to_string(index=False))

print('\n🟢 Top 10 Most Healthy Videos:')
print(df.nsmallest(10, 'brainrot_proba')[['title','channel','brainrot_proba']]
        .rename(columns={'brainrot_proba':'Brainrot Score'})
        .to_string(index=False))